In [ ]:
## Stage Dataset Assembly

# Assemble datasets for each training stage
def assemble_stage_datasets():
    """Assemble datasets for each training stage."""
    print("Assembling stage datasets...")
    
    # Stage 1: Akkademia + ORACC
    # Combine Akkademia train and valid sets (test is held out)
    akkademia_stage1 = pd.concat([
        akkademia_deduped['train'], 
        akkademia_deduped['valid']
    ])
    
    # Add ORACC data
    # Weight HIGH-genre ORACC rows 2×, LOW-genre 0.5×
    # For simplicity, we'll just add all ORACC data to start
    oracc_stage1 = oracc_deduped.copy()
    
    # Combine datasets
    stage1_data = pd.concat([
        akkademia_stage1[['transliteration_normalized', 'translation_normalized']].rename(
            columns={'transliteration_normalized': 'transliteration', 'translation_normalized': 'translation'}
        ),
        oracc_stage1[['transliteration_normalized', 'translation_normalized']].rename(
            columns={'transliteration_normalized': 'transliteration', 'translation_normalized': 'translation'}
        )
    ], ignore_index=True)
    
    print(f"Stage 1 dataset assembled: {len(stage1_data)} rows")
    
    # Stage 2: Sentences_Oare extra
    # Rows NOT in train tablets. Drop null translations.
    stage2_data = sentences_oare_filtered[['text_uuid', 'translation', 'first_word_spelling', 'line_number', 'side']].copy()
    stage2_data = stage2_data.rename(columns={'translation': 'translation_raw'})
    
    # We need to merge with the transliterations from another source
    # For now, let's just use what we have and note that we need to get transliterations
    print(f"Stage 2 dataset prepared: {len(stage2_data)} rows (need to add transliterations)")
    
    # Stage 3: competition train
    # First 90% by index (preserve document order, no random split)
    stage3_full = train_with_genre[['transliteration_prefixed', 'translation_normalized']].rename(
        columns={'transliteration_prefixed': 'transliteration', 'translation_normalized': 'translation'}
    )
    
    split_idx = int(len(stage3_full) * 0.9)
    stage3_train = stage3_full.iloc[:split_idx].copy()
    stage3_val = stage3_full.iloc[split_idx:].copy()
    
    print(f"Stage 3 train dataset: {len(stage3_train)} rows")
    print(f"Stage 3 validation dataset: {len(stage3_val)} rows")
    
    # Save all datasets
    stage1_data.to_csv('data/stage1_train.csv', index=False)
    stage2_data.to_csv('data/stage2_train.csv', index=False)
    stage3_train.to_csv('data/stage3_train.csv', index=False)
    stage3_val.to_csv('data/stage3_val.csv', index=False)
    
    print("Stage datasets saved to data/")
    
    return stage1_data, stage2_data, stage3_train, stage3_val

# Assemble stage datasets
stage1_data, stage2_data, stage3_train, stage3_val = assemble_stage_datasets()

In [ ]:
## Build Glossary File

# Create glossary mapping for proper names
def build_glossary():
    """Build glossary file from OA_Lexicon_eBL.csv."""
    print("Building glossary...")
    
    # From OA_Lexicon_eBL.csv: create a dict mapping normalized form → norm
    # Filter for personal names and place names
    names_df = lexicon_df[lexicon_df['type'].isin(['PN', 'GN', 'MN'])].copy()  # Personal, Geographic, Month names
    
    # Create dictionary mapping
    glossary_dict = dict(zip(names_df['form'], names_df['norm']))
    
    print(f"Glossary created with {len(glossary_dict)} entries")
    print("Sample entries:", dict(list(glossary_dict.items())[:5]))
    
    # Save as data/glossary.json
    with open('data/glossary.json', 'w', encoding='utf-8') as f:
        json.dump(glossary_dict, f, ensure_ascii=False, indent=2)
    
    print("Glossary saved to data/glossary.json")
    
    return glossary_dict

# Build glossary
glossary_dict = build_glossary()

In [ ]:
## Genre Prefix Conditioning

# Apply genre prefix conditioning
def apply_genre_prefix_conditioning():
    """Apply genre prefix conditioning to training data."""
    print("Applying genre prefix conditioning...")
    
    # Merge published_texts.csv into train.csv on oare_id to get genre_label
    train_with_genre = train_df.merge(
        published_texts_df[['oare_id', 'genre_label']], 
        on='oare_id', 
        how='left'
    )
    
    # Map each row to a prefix token
    def map_genre_to_prefix(genre_label):
        if pd.isna(genre_label):
            return '[UNKNOWN]'
        
        genre_mapping = {
            'letter': '[LETTER]',
            'debt note': '[DEBT_NOTE]',
            'contract': '[CONTRACT]',
            'administrative': '[ADMIN]',
        }
        
        # Convert to lowercase for matching
        genre_lower = genre_label.lower().strip()
        
        # Try exact match first
        for key, prefix in genre_mapping.items():
            if key in genre_lower:
                return prefix
                
        # Try partial match
        if 'letter' in genre_lower:
            return '[LETTER]'
        elif 'debt' in genre_lower or 'loan' in genre_lower:
            return '[DEBT_NOTE]'
        elif 'contract' in genre_lower:
            return '[CONTRACT]'
        elif 'admin' in genre_lower:
            return '[ADMIN]'
        else:
            return '[UNKNOWN]'
    
    # Apply mapping
    train_with_genre['genre_prefix'] = train_with_genre['genre_label'].apply(map_genre_to_prefix)
    
    # Prepend prefix to transliteration
    train_with_genre['transliteration_prefixed'] = (
        train_with_genre['genre_prefix'] + ' ' + train_with_genre['transliteration_normalized']
    )
    
    print("Genre prefix conditioning applied!")
    print("Genre distribution:")
    print(train_with_genre['genre_prefix'].value_counts())
    
    return train_with_genre

# Apply genre prefix conditioning
train_with_genre = apply_genre_prefix_conditioning()

In [ ]:
## Cross-Dataset Deduplication

# Perform cross-dataset deduplication
def deduplicate_datasets():
    """Remove duplicate entries across datasets."""
    print("Performing cross-dataset deduplication...")
    
    # Build a set of all normalized competition transliteration values
    competition_transliterations = set(train_df['transliteration_normalized'].dropna())
    print(f"Competition train set has {len(competition_transliterations)} unique transliterations")
    
    # Remove ANY row from Akkademia or ORACC that exactly matches a competition training row
    akkademia_deduped = {}
    for split in ['train', 'valid', 'test']:
        before_count = len(akkademia_data[split])
        akkademia_deduped[split] = akkademia_data[split][
            ~akkademia_data[split]['transliteration_normalized'].isin(competition_transliterations)
        ].copy()
        after_count = len(akkademia_deduped[split])
        print(f"Akkademia {split}: {before_count} -> {after_count} rows (removed {before_count - after_count})")
    
    # Deduplicate within each dataset on transliteration after normalization
    print("\nDeduplicating within datasets...")
    
    # Deduplicate Akkademia data
    for split in ['train', 'valid', 'test']:
        before_count = len(akkademia_deduped[split])
        akkademia_deduped[split] = akkademia_deduped[split].drop_duplicates(
            subset=['transliteration_normalized'], keep='first'
        )
        after_count = len(akkademia_deduped[split])
        print(f"Akkademia {split} dedup: {before_count} -> {after_count} rows (removed {before_count - after_count})")
    
    # Deduplicate ORACC data
    before_count = len(oracc_df)
    oracc_deduped = oracc_df.drop_duplicates(
        subset=['transliteration_normalized'], keep='first'
    )
    after_count = len(oracc_deduped)
    print(f"ORACC dedup: {before_count} -> {after_count} rows (removed {before_count - after_count})")
    
    # Extract Sentences_Oare rows whose text_uuid is NOT in train.oare_id
    sentences_oare_filtered = sentences_oare_df[
        ~sentences_oare_df['text_uuid'].isin(train_df['oare_id'])
    ].copy()
    print(f"Sentences_Oare: {len(sentences_oare_df)} -> {len(sentences_oare_filtered)} rows (filtered out competition train texts)")
    
    # Drop null translations from Sentences_Oare
    before_count = len(sentences_oare_filtered)
    sentences_oare_filtered = sentences_oare_filtered.dropna(subset=['translation'])
    after_count = len(sentences_oare_filtered)
    print(f"Sentences_Oare null translations removed: {before_count} -> {after_count} rows")
    
    print("Cross-dataset deduplication complete!")
    
    return akkademia_deduped, oracc_deduped, sentences_oare_filtered

# Perform deduplication
akkademia_deduped, oracc_deduped, sentences_oare_filtered = deduplicate_datasets()

In [ ]:
## Data Preprocessing

# Define preprocessing functions
def normalize_transliteration(text):
    """Normalize transliteration text according to specifications."""
    if pd.isna(text) or text == "":
        return text
    
    # Unicode NFC normalization
    text = unicodedata.normalize('NFC', text)
    
    # Map Unicode subscript digits to regular ASCII digits
    subscript_map = {'₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4', 
                     '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9'}
    for sub, norm in subscript_map.items():
        text = text.replace(sub, norm)
    
    # Replace damage markers [...] and broken text sequences with <gap>
    text = re.sub(r'\[.*?\]', '<gap>', text)
    
    # Strip scribal correction markers: !, ?, #
    text = re.sub(r'[!?#]', '', text)
    
    # Preserve determinatives in curly braces like {d}, {f}
    # Preserve Sumerian logograms in uppercase
    
    # Collapse multiple spaces, strip leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def normalize_translation(text):
    """Normalize translation text according to specifications."""
    if pd.isna(text) or text == "":
        return text
    
    # Normalize quotation marks to standard ASCII
    text = re.sub(r'[“”‟ˮ＂]', '"', text)
    text = re.sub(r"[‘’‛ʼ՚]", "'", text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Strip editorial bracket marks but keep content inside
    text = re.sub(r'^[\[\]()]+|[\[\]()]+$', '', text)
    
    return text

# Apply preprocessing to all datasets
def preprocess_datasets():
    """Apply normalization to all datasets."""
    print("Applying preprocessing to datasets...")
    
    # Preprocess competition train data
    train_df['transliteration_normalized'] = train_df['transliteration'].apply(normalize_transliteration)
    train_df['translation_normalized'] = train_df['translation'].apply(normalize_translation)
    
    # Preprocess Akkademia data
    for split in ['train', 'valid', 'test']:
        akkademia_data[split]['transliteration_normalized'] = akkademia_data[split]['transliteration'].apply(normalize_transliteration)
        akkademia_data[split]['translation_normalized'] = akkademia_data[split]['translation'].apply(normalize_translation)
    
    # Preprocess ORACC data (assuming columns are named 'akkadian' and 'english')
    # We need to identify the correct column names first
    oracc_columns = oracc_df.columns.tolist()
    print(f"ORACC columns: {oracc_columns}")
    
    # Try to identify Akkadian and English columns
    akkadian_col = None
    english_col = None
    
    for col in oracc_columns:
        col_lower = col.lower()
        if 'akkad' in col_lower or 'translit' in col_lower:
            akkadian_col = col
        elif 'eng' in col_lower or 'translat' in col_lower:
            english_col = col
    
    print(f"Identified columns - Akkadian: {akkadian_col}, English: {english_col}")
    
    if akkadian_col and english_col:
        oracc_df['transliteration_normalized'] = oracc_df[akkadian_col].apply(normalize_transliteration)
        oracc_df['translation_normalized'] = oracc_df[english_col].apply(normalize_translation)
    
    print("Preprocessing complete!")
    
    return train_df, akkademia_data, oracc_df

# Apply preprocessing
train_df, akkademia_data, oracc_df = preprocess_datasets()

In [ ]:
## Load and Classify ORACC Kaggle Dataset

# Load and classify ORACC Kaggle dataset
def load_and_classify_oracc():
    """Load and classify the ORACC Kaggle dataset based on genre priority."""
    print("Loading ORACC Kaggle dataset...")
    oracc_df = pd.read_csv('data/oracc_kaggle/train.csv')
    
    print(f"ORACC dataset loaded successfully! Shape: {oracc_df.shape}")
    print("ORACC columns:", oracc_df.columns.tolist())
    
    # Check if there's a genre/corpus column
    genre_columns = [col for col in oracc_df.columns if 'genre' in col.lower() or 'corpus' in col.lower()]
    print(f"Potential genre columns: {genre_columns}")
    
    # Display some sample data
    print("\nFirst few rows of ORACC dataset:")
    print(oracc_df.head())
    
    # If we have genre information, classify by priority
    genre_priority = {
        'HIGH': ['letter', 'administrative', 'memo', 'note', 'decree', 'report'],
        'MEDIUM': ['legal', 'contract', 'list'],
        'LOW': ['royal inscription', 'votive inscription', 'building inscription', 'annals']
    }
    
    print("\nGenre priority classification:")
    for priority, genres in genre_priority.items():
        print(f"{priority}: {genres}")
    
    return oracc_df

# Load and classify ORACC dataset
oracc_df = load_and_classify_oracc()

In [ ]:
## Parse Akkademia NMT_input

# Parse Akkademia NMT_input files
def parse_akkademia_nmt_input():
    """Parse the Akkademia NMT_input files and create aligned datasets."""
    akkademia_train = akkademia_data['train']
    akkademia_valid = akkademia_data['valid']
    akkademia_test = akkademia_data['test']
    
    print("Akkademia NMT_input parsed successfully!")
    print(f"Train set: {len(akkademia_train)} pairs")
    print(f"Validation set: {len(akkademia_valid)} pairs")
    print(f"Test set: {len(akkademia_test)} pairs")
    
    # Save parsed data
    akkademia_train.to_csv('data/akkademia/akkademia_train_parsed.csv', index=False)
    akkademia_valid.to_csv('data/akkademia/akkademia_valid_parsed.csv', index=False)
    akkademia_test.to_csv('data/akkademia/akkademia_test_parsed.csv', index=False)
    
    print("Parsed Akkademia data saved to data/akkademia/")
    
    return akkademia_train, akkademia_valid, akkademia_test

# Parse the Akkademia NMT_input
akkademia_train, akkademia_valid, akkademia_test = parse_akkademia_nmt_input()

# Deep Past Initiative: Akkadian → English Machine Translation

This notebook implements a 3-stage neural machine translation pipeline for translating Old Assyrian Akkadian transliterations into English.

In [ ]:
# Project initialization and setup
import os
import sys
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
import unicodedata
import re
import json

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('models/stage1_final', exist_ok=True)
os.makedirs('models/stage2_final', exist_ok=True)
os.makedirs('models/stage3_final', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("Project structure initialized successfully!")

## Data Loading and Initial Exploration

In [ ]:
# Load competition datasets
train_df = pd.read_csv('data/competition/train.csv')
test_df = pd.read_csv('data/competition/test.csv')
sample_submission_df = pd.read_csv('data/competition/sample_submission.csv')
sentences_oare_df = pd.read_csv('data/competition/Sentences_Oare_FirstWord_LinNum.csv')
published_texts_df = pd.read_csv('data/competition/published_texts.csv')
lexicon_df = pd.read_csv('data/competition/OA_Lexicon_eBL.csv')
dictionary_df = pd.read_csv('data/competition/eBL_Dictionary.csv')

print("Competition datasets loaded successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sample submission shape: {sample_submission_df.shape}")
print(f"Sentences_Oare shape: {sentences_oare_df.shape}")
print(f"Published texts shape: {published_texts_df.shape}")
print(f"Lexicon shape: {lexicon_df.shape}")
print(f"Dictionary shape: {dictionary_df.shape}")

## Load External Datasets

In [ ]:
# Load Akkademia dataset
def load_akkademia_dataset():
    akkademia_data = {}
    splits = ['train', 'valid', 'test']
    
    for split in splits:
        with open(f'data/akkademia/{split}.ak', 'r', encoding='utf-8') as f:
            akkadian_lines = [line.strip() for line in f.readlines()]
        with open(f'data/akkademia/{split}.en', 'r', encoding='utf-8') as f:
            english_lines = [line.strip() for line in f.readlines()]
            
        akkademia_data[split] = pd.DataFrame({
            'transliteration': akkadian_lines,
            'translation': english_lines
        })
        
    print("Akkademia dataset loaded successfully!")
    for split in splits:
        print(f"{split} set size: {len(akkademia_data[split])}")
        
    return akkademia_data

# Load ORACC dataset
oracc_df = pd.read_csv('data/oracc_kaggle/train.csv')
print(f"ORACC dataset loaded successfully! Shape: {oracc_df.shape}")
print("ORACC columns:", oracc_df.columns.tolist())

# Load Akkademia data
akkademia_data = load_akkademia_dataset()